In [2]:
import torch
print("¿PyTorch detecta CUDA/GPU?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Nombre de la GPU:", torch.cuda.get_device_name(0))
else:
    print("ADVERTENCIA: Estás ejecutando en CPU (por eso va tan lento).")

¿PyTorch detecta CUDA/GPU?: False
ADVERTENCIA: Estás ejecutando en CPU (por eso va tan lento).


In [ ]:
import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from datasets import Dataset
from sklearn.metrics import classification_report, precision_recall_fscore_support
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer, 
    DataCollatorWithPadding
)

# 1. Verificar dispositivo (GPU o CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo detectado para entrenamiento: {device}")

# 2. Cargar conjuntos de datos locales (ubicados en la carpeta ../data/)
train_df = pd.read_csv('../data/train.csv')
val_df = pd.read_csv('../data/val.csv')
test_df = pd.read_csv('../data/test.csv')

# Convertir a Hugging Face Dataset
train_dataset = Dataset.from_pandas(train_df[['texto_modelo', 'label']])
val_dataset = Dataset.from_pandas(val_df[['texto_modelo', 'label']])
test_dataset = Dataset.from_pandas(test_df[['texto_modelo', 'label']])

# 3. Tokenización con XLM-RoBERTa
MODEL_NAME = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples['texto_modelo'], truncation=True, max_length=128)

train_tok = train_dataset.map(tokenize_function, batched=True)
val_tok = val_dataset.map(tokenize_function, batched=True)
test_tok = test_dataset.map(tokenize_function, batched=True)

# 4. Trainer con Pérdida Ponderada
class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = torch.tensor(class_weights, dtype=torch.float32)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        weights = self.class_weights.to(logits.device)
        loss_fct = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro')
    acc = (preds == labels).mean()
    return {
        'f1_macro': f1,
        'precision_macro': precision,
        'recall_macro': recall,
        'accuracy': acc
    }

# 5. Configurar Modelo y Entrenamiento
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=4)
class_weights = [0.9404, 0.7224, 1.2099, 1.3779]

# fp16 solo se activa si hay una GPU NVIDIA disponible
use_fp16 = torch.cuda.is_available()

training_args = TrainingArguments(
    output_dir='../results_xlmr',
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=25,
    fp16=use_fp16,
    seed=42
)

trainer = WeightedTrainer(
    class_weights=class_weights,
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

# 6. Entrenar en local
trainer.train()

# 7. Evaluación en Test
labels_names = ['No Tóxico', 'Lenguaje Ofensivo', 'Discurso de Odio', 'Amenazas/Violencia']
test_results = trainer.predict(test_tok)
test_preds = np.argmax(test_results.predictions, axis=1)

print("=== Reporte de Clasificación Global ===")
print(classification_report(test_df['label'], test_preds, target_names=labels_names))